# IMGDS Experiment 1: Progressive Attention-Only Q/K Active-Channel Sparse
## Conservative, Gradual Sparsity: 5% → 10% → 15% → 20% → 30%

**Principle:** Only prune Q/K channels in Linear Attention. input_projection, feedforward, classifier, position_embedding, V, and output projection stay intact.

**Environment:** Docker Jupyter Kernel (http://127.0.0.1:8886)

**RULES:**
- NO retraining
- NO modifying src/transformer.py
- NO global channel pruning
- NO fabricating results
- NO 50%/70%/90%/100% sparse in first HLS batch
- YES: attention-only Q/K active-channel sparse
- YES: progressive sparsity (5/10/15/20/30%)
- YES: conservative HLS candidate selection


---
## Cell 1: Kernel Verification
---

In [ ]:
import sys, os, json, csv, time, warnings, copy
import numpy as np
from collections import OrderedDict

print(f'Python: {sys.executable}')
exec_lower = sys.executable.lower()
assert 'anaconda' not in exec_lower and 'conda' not in exec_lower, \
    f'ERROR: Host conda Python ({sys.executable}). Switch to Docker kernel!'
print('OK: Docker/Jupyter kernel confirmed.')

import torch
import torch.nn as nn
import torch.nn.functional as F
print(f'PyTorch {torch.__version__}')

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
print('All imports OK.')


---
## Cell 2: Paths, Constants, and Model Config
---

In [ ]:
PROJ = '/home/cym/prj2/finn/notebooks/icl_thesis-master'
EXP  = f'{PROJ}/experiments/imgds_linear_sparse'
FP   = f'{EXP}/final_paper_experiments'
TABLES = f'{FP}/tables'
CHECKPOINT_DIR = f'{EXP}/checkpoints'
OUTPUTS_DIR = f'{EXP}/outputs'

os.makedirs(TABLES, exist_ok=True)

# Model constants
D_MODEL = 16
SEQ_LEN = 16
INPUT_DIM = 64
FF_DIM = 32
N_CLASSES = 2

MODEL_CONFIG = {
    'input_dim': INPUT_DIM, 'seq_len': SEQ_LEN, 'd_model': D_MODEL,
    'dim_feedforward': FF_DIM, 'num_layers': 1, 'num_classes': N_CLASSES, 'dropout': 0.0,
}

# Progressive sparsity levels
Q16_SPARSITY = [0, 5, 10, 15, 20, 30]
Q8_SPARSITY  = [0, 5, 10, 15, 20]

def sparsity_to_active_dim(sp_pct):
    mapping = {0: 16, 5: 15, 10: 14, 15: 14, 20: 13, 30: 11}
    return mapping.get(sp_pct, max(round(D_MODEL * (1 - sp_pct/100.0)), 1))

print('Paths and constants ready.')
print(f'Q16 sweep: {Q16_SPARSITY}')
print(f'Q8 sweep:  {Q8_SPARSITY}')
for sp in sorted(set(Q16_SPARSITY + Q8_SPARSITY)):
    ad = sparsity_to_active_dim(sp)
    print(f'  S={sp:2d}% -> active_dim={ad}')


---
## Cell 3: Load Trained Checkpoint and Test Data
---

**NO RETRAINING.** Load existing weights. Load full test set (3,444 samples).


In [ ]:
sys.path.insert(0, f'{PROJ}/src')
from transformer import LinearUNSWAnomalyDetector, _UNSWLinearAttention

CKPT_PATH = f'{CHECKPOINT_DIR}/best_linear_imgds_dense_r32_p8.pt'
assert os.path.exists(CKPT_PATH), f'Checkpoint not found: {CKPT_PATH}'

ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
if 'state_dict' in ckpt:
    state_dict = ckpt['state_dict']
elif 'model_state_dict' in ckpt:
    state_dict = ckpt['model_state_dict']
else:
    state_dict = ckpt

total_params = sum(v.numel() for v in state_dict.values())
print(f'Loaded checkpoint: {total_params} parameters')
for k, v in state_dict.items():
    print(f'  {k:<60s} shape={str(v.shape):<20s}')

TEST_NPZ = f'{OUTPUTS_DIR}/imgds_r32_p8_test.npz'
test_data = np.load(TEST_NPZ)
X_test = test_data['X']  # (3444, 16, 64)
y_test = test_data['y']  # (3444,)
n_test = len(y_test)
print(f'\nTest data: X={X_test.shape}, y={y_test.shape}')
print(f'Labels: benign={np.sum(y_test==0)}, malware={np.sum(y_test==1)}')


---
## Cell 4: Q/K Channel Importance (Attention Weights Only)
---

**Key difference from global pruning:** Only use query.weight and key.weight L1 norms.
Do NOT include feedforward, classifier, input_projection weights.
This ensures we select channels important specifically for attention computation.


In [ ]:
def compute_qk_channel_importance(state_dict):
    '''
    Compute channel importance using ONLY attention Q/K weights.
    Uses row-wise L1 norm: |Wq[d,:]| + |Wk[d,:]| for each channel d.
    Returns: array of shape (D_MODEL,) — higher = more important.
    '''
    importance = np.zeros(D_MODEL)
    for prefix in ['backbone.layers.0.attention.query.weight',
                   'backbone.layers.0.attention.key.weight']:
        w = state_dict[prefix].numpy()  # [D_MODEL, D_MODEL]
        importance += np.abs(w).sum(axis=1)  # row norm = output channel importance
    return importance

def select_active_channels_qk(importance, active_dim):
    if active_dim >= D_MODEL:
        return list(range(D_MODEL))
    if active_dim <= 0:
        return []
    sorted_idx = np.argsort(importance)[::-1]
    return sorted(sorted_idx[:active_dim].tolist())

# Compute Q/K channel importance
qk_importance = compute_qk_channel_importance(state_dict)
sorted_idx = np.argsort(qk_importance)[::-1]

print('Q/K Channel Importance Ranking (L1 norm of query+key weight rows):')
print(f'  {"Rank":<6} {"Ch":<5} {"Importance":<12} {"Keep?"}')
for rank, ch in enumerate(sorted_idx):
    keep = 'KEEP' if rank < 14 else ('borderline' if rank < 16 else 'DROP')
    print(f'  {rank+1:<6} {ch:<5} {qk_importance[ch]:<12.4f} {keep}')

# Pre-compute active channels for each sparsity level
all_sparsity = sorted(set(Q16_SPARSITY + Q8_SPARSITY))
qk_active_cache = {}
for sp in all_sparsity:
    active_dim = sparsity_to_active_dim(sp)
    qk_active_cache[sp] = select_active_channels_qk(qk_importance, active_dim)
    inactive = [c for c in range(D_MODEL) if c not in qk_active_cache[sp]]
    print(f'  S={sp:2d}% -> active_dim={active_dim:2d} -> inactive={inactive}')


---
## Cell 5: Attention-Only Q/K Active-Channel Sparse Monkey-Patch
---

**Method:** Monkey-patch `_UNSWLinearAttention.forward()` to zero out Q/K for inactive channels AFTER feature_map.
This simulates HLS truncated Q/K loops without modifying src/transformer.py.

**What is pruned:** Only Q and K feature channels.
**What stays intact:** V, output projection, input_projection, feedforward, classifier, position_embedding, LayerNorms.

**HLS equivalent:**
```cpp
for (int ii = 0; ii < ACTIVE_DIM; ii++) {
    int c = active_idx[ii];
    // Compute Q[s][c] and K[s][c] only for active channels
}
```


In [ ]:
def make_attention_only_qk_sparse_forward(original_forward, active_channels, d_model=D_MODEL):
    '''
    Returns a patched forward for _UNSWLinearAttention that zeroes out
    Q and K for inactive channels AFTER feature_map.
    '''
    inactive_mask = torch.zeros(d_model)
    for c in active_channels:
        inactive_mask[c] = 1.0
    
    def sparse_forward(self, x):
        # Original projections (all channels)
        query = self._feature_map(self.query(x))   # [B, S, D]
        key   = self._feature_map(self.key(x))      # [B, S, D]
        value = self.value(x)                        # [B, S, D] — KEPT FULL
        
        # === ATTENTION-ONLY Q/K SPARSE ===
        # Zero out inactive Q/K channels to simulate HLS truncated loops
        mask = inactive_mask.to(x.device)
        query_sparse = query * mask[None, None, :]   # inactive=0
        key_sparse   = key   * mask[None, None, :]   # inactive=0
        # V stays complete
        
        # Kernelized attention (same as original, with sparse Q/K)
        key_value = torch.matmul(key_sparse.transpose(-2, -1), value)
        key_sum = key_sparse.sum(dim=1, keepdim=False).unsqueeze(-1)
        normalizer = torch.matmul(query_sparse, key_sum).clamp_min(self.eps)
        attended = torch.matmul(query_sparse, key_value) / normalizer
        
        return self.output(attended)  # output projection — KEPT FULL
    
    return sparse_forward


def run_inference_attention_sparse(model, X, active_channels, batch_size=512):
    '''
    Run inference with attention-only Q/K sparse.
    Monkey-patches _UNSWLinearAttention.forward() temporarily.
    '''
    attn_module = model.layers[0].attention
    original_forward = attn_module.forward
    
    # Patch with sparse forward
    attn_module.forward = make_attention_only_qk_sparse_forward(
        original_forward, active_channels, D_MODEL
    ).__get__(attn_module, type(attn_module))
    
    try:
        all_logits = []
        all_preds = []
        with torch.no_grad():
            for i in range(0, len(X), batch_size):
                batch = torch.from_numpy(X[i:i+batch_size]).float()
                logits = model(batch)
                preds = torch.argmax(logits, dim=1)
                all_logits.append(logits.numpy())
                all_preds.append(preds.numpy())
        return np.concatenate(all_logits, axis=0), np.concatenate(all_preds, axis=0)
    finally:
        # ALWAYS restore
        attn_module.forward = original_forward


print('Attention-only Q/K sparse monkey-patch ready.')

# Quick sanity check
model_test = LinearUNSWAnomalyDetector(**MODEL_CONFIG)
model_test.load_state_dict(state_dict, strict=True)
model_test.eval()

# Dense baseline (all channels active)
logits_dense, preds_dense = run_inference_attention_sparse(
    model_test, X_test[:100], list(range(D_MODEL)))
n_dense = len(preds_dense)
print(f'Dense baseline (first 100): preds={np.bincount(preds_dense)}')

# Sparse test (14 active channels = S10%)
active_14 = qk_active_cache[10]
logits_14, preds_14 = run_inference_attention_sparse(
    model_test, X_test[:100], active_14)
match_rate = (preds_dense == preds_14).mean()
n_match = int(match_rate * n_dense)
print(f'S=10% (active={active_14}) vs dense match: {match_rate:.4f} ({n_match}/{n_dense})')
print('Sanity check OK.')


---
## Cell 6: Fake Quantization Functions
---

In [ ]:
def fake_quantize_tensor(w, total_bits, int_bits):
    if total_bits == 32:
        return w.clone()
    frac_bits = total_bits - int_bits
    max_val = 2**(int_bits - 1) - 2**(-frac_bits)
    min_val = -2**(int_bits - 1)
    scale = 2**frac_bits
    w_scaled = w * scale
    w_rounded = torch.round(w_scaled)
    w_clamped = torch.clamp(w_rounded, min_val * scale, max_val * scale)
    return w_clamped / scale

def fake_quantize_state_dict(state_dict, total_bits):
    if total_bits == 32:
        return OrderedDict((k, v.clone()) for k, v in state_dict.items())
    quantized = OrderedDict()
    for k, v in state_dict.items():
        is_weight = 'weight' in k and v.ndim >= 2
        is_norm = 'norm' in k.lower()
        if is_weight and not is_norm:
            if total_bits == 16:
                quantized[k] = fake_quantize_tensor(v, 16, 6)
            elif total_bits == 8:
                quantized[k] = fake_quantize_tensor(v, 8, 4)
            elif total_bits == 4:
                quantized[k] = fake_quantize_tensor(v, 4, 2)
            else:
                quantized[k] = v.clone()
        else:
            quantized[k] = v.clone()
    return quantized

# Pre-compute quantized state_dicts
print('Pre-computing quantized state_dicts...')
q16_sd = fake_quantize_state_dict(state_dict, 16)
q8_sd  = fake_quantize_state_dict(state_dict, 8)
q32_sd = OrderedDict((k, v.clone()) for k, v in state_dict.items())
print('Q32, Q16, Q8 state_dicts ready.')


---
## Cell 7: Q16 Progressive Attention-Only Sparse Sweep
---

**This is the main experiment.** Run Q16 with sparsity 0%, 5%, 10%, 15%, 20%, 30%.
Only Q/K channels are pruned. Full test set (3,444 samples).


In [ ]:
import time as time_module

def compute_metrics_full(y_true, y_pred, logits, ref_logits=None):
    metrics = {}
    metrics['accuracy'] = accuracy_score(y_true, y_pred)
    metrics['precision'] = precision_score(y_true, y_pred, zero_division=0)
    metrics['recall'] = recall_score(y_true, y_pred, zero_division=0)
    metrics['f1'] = f1_score(y_true, y_pred, zero_division=0)
    probs = torch.softmax(torch.from_numpy(logits), dim=1).numpy()
    metrics['auc'] = roc_auc_score(y_true, probs[:, 1]) if len(np.unique(y_true)) > 1 else 0.5
    if ref_logits is not None:
        ref_preds = np.argmax(ref_logits, axis=1)
        metrics['prediction_match_rate_vs_q32'] = (y_pred == ref_preds).mean()
        logit_errors = np.abs(logits - ref_logits)
        metrics['max_logit_error_vs_q32'] = logit_errors.max()
        metrics['mean_logit_error_vs_q32'] = logit_errors.mean()
    metrics['num_samples'] = len(y_true)
    return metrics

print('=' * 80)
print('Q16 ATTENTION-ONLY Q/K PROGRESSIVE SPARSE SWEEP')
print('=' * 80)

# Baseline Q32 dense for reference
model_q32 = LinearUNSWAnomalyDetector(**MODEL_CONFIG)
model_q32.load_state_dict(q32_sd, strict=True)
model_q32.eval()
logits_q32, preds_q32 = run_inference_attention_sparse(model_q32, X_test, list(range(D_MODEL)))
q32_acc = accuracy_score(y_test, preds_q32)
print(f'Q32 baseline: acc={q32_acc:.4f}')

# Q16 sweep
q16_results = []
model_q16 = LinearUNSWAnomalyDetector(**MODEL_CONFIG)
model_q16.load_state_dict(q16_sd, strict=True)
model_q16.eval()

for sp in Q16_SPARSITY:
    active_dim = sparsity_to_active_dim(sp)
    active_ch = qk_active_cache[sp]
    eid = f'e1_q16_s{sp}_attn_only'
    
    t0 = time_module.time()
    logits, preds = run_inference_attention_sparse(model_q16, X_test, active_ch)
    metrics = compute_metrics_full(y_test, preds, logits, ref_logits=logits_q32)
    dt = time_module.time() - t0
    
    result = {
        'experiment_id': eid,
        'quant_bits': 16,
        'sparsity_percent': sp,
        'active_dim': active_dim,
        'active_channels': str(active_ch),
        'accuracy': round(metrics['accuracy'], 6),
        'precision': round(metrics['precision'], 6),
        'recall': round(metrics['recall'], 6),
        'f1': round(metrics['f1'], 6),
        'auc': round(metrics['auc'], 6),
        'prediction_match_rate_vs_q32': round(metrics.get('prediction_match_rate_vs_q32', 1.0), 6),
        'max_logit_error_vs_q32': round(metrics.get('max_logit_error_vs_q32', 0.0), 6),
        'mean_logit_error_vs_q32': round(metrics.get('mean_logit_error_vs_q32', 0.0), 6),
        'sparse_type': 'attention_only_qk_active_channel',
        'status': 'SUCCESS',
        'notes': '',
    }
    q16_results.append(result)
    
    acc_pct = metrics['accuracy'] * 100
    vs_vit4mal = 'ABOVE ViT4Mal' if acc_pct >= 92.41 else 'BELOW ViT4Mal'
    print(f'  [{eid}] S={sp:2d}% act_dim={active_dim:2d} '
          f'acc={acc_pct:.2f}% match={metrics["prediction_match_rate_vs_q32"]:.4f} '
          f'f1={metrics["f1"]:.4f} ({dt:.1f}s) {vs_vit4mal}')

n_above = sum(1 for r in q16_results if r['accuracy'] >= 0.9241)
n_above_rec = sum(1 for r in q16_results if r['accuracy'] >= 0.9389)
print(f'\nQ16 sweep complete: {len(q16_results)} points.')
print(f'Above ViT4Mal fastest (92.41%): {n_above}')
print(f'Above ViT4Mal rec (93.89%):      {n_above_rec}')


---
## Cell 8: Q8 Progressive Attention-Only Sparse Sweep
---

Run Q8 with conservative sparsity: 0%, 5%, 10%, 15%, 20%.


In [ ]:
q16_acc_0  = [r for r in q16_results if r['sparsity_percent'] == 0][0]['accuracy']
q16_acc_10 = [r for r in q16_results if r['sparsity_percent'] == 10][0]['accuracy']
q16_acc_drop = (q16_acc_0 - q16_acc_10) * 100

print(f'Q16 S0 acc:  {q16_acc_0:.4f}')
print(f'Q16 S10 acc: {q16_acc_10:.4f}')
print(f'Q16 S0->S10 drop: {q16_acc_drop:.2f} pp')

if q16_acc_drop > 5.0:
    print(f'\nWARNING: Q16 attention-only S10 drops {q16_acc_drop:.1f}pp. Flagging for review.')

print('\n' + '=' * 80)
print('Q8 ATTENTION-ONLY Q/K PROGRESSIVE SPARSE SWEEP')
print('=' * 80)

q8_results = []
model_q8 = LinearUNSWAnomalyDetector(**MODEL_CONFIG)
model_q8.load_state_dict(q8_sd, strict=True)
model_q8.eval()

for sp in Q8_SPARSITY:
    active_dim = sparsity_to_active_dim(sp)
    active_ch = qk_active_cache[sp]
    eid = f'e1_q8_s{sp}_attn_only'
    
    t0 = time_module.time()
    logits, preds = run_inference_attention_sparse(model_q8, X_test, active_ch)
    metrics = compute_metrics_full(y_test, preds, logits, ref_logits=logits_q32)
    dt = time_module.time() - t0
    
    result = {
        'experiment_id': eid,
        'quant_bits': 8,
        'sparsity_percent': sp,
        'active_dim': active_dim,
        'active_channels': str(active_ch),
        'accuracy': round(metrics['accuracy'], 6),
        'precision': round(metrics['precision'], 6),
        'recall': round(metrics['recall'], 6),
        'f1': round(metrics['f1'], 6),
        'auc': round(metrics['auc'], 6),
        'prediction_match_rate_vs_q32': round(metrics.get('prediction_match_rate_vs_q32', 1.0), 6),
        'max_logit_error_vs_q32': round(metrics.get('max_logit_error_vs_q32', 0.0), 6),
        'mean_logit_error_vs_q32': round(metrics.get('mean_logit_error_vs_q32', 0.0), 6),
        'sparse_type': 'attention_only_qk_active_channel',
        'status': 'SUCCESS',
        'notes': '',
    }
    q8_results.append(result)
    
    acc_pct = metrics['accuracy'] * 100
    vs_vit4mal = 'ABOVE' if acc_pct >= 92.41 else 'BELOW'
    print(f'  [{eid}] S={sp:2d}% act_dim={active_dim:2d} '
          f'acc={acc_pct:.2f}% match={metrics["prediction_match_rate_vs_q32"]:.4f} '
          f'f1={metrics["f1"]:.4f} ({dt:.1f}s) {vs_vit4mal} ViT4Mal')

n_q8_above = sum(1 for r in q8_results if r['accuracy'] >= 0.9241)
print(f'\nQ8 sweep complete: {len(q8_results)} points, {n_q8_above} above ViT4Mal fastest.')


---
## Cell 9: Save Progressive Attention-Only Results
---

In [ ]:
all_results = q16_results + q8_results

results_csv = f'{TABLES}/experiment1_attention_only_progressive_sparse_results.csv'
fieldnames = ['experiment_id','quant_bits','sparsity_percent','active_dim',
              'active_channels','accuracy','precision','recall','f1','auc',
              'prediction_match_rate_vs_q32','max_logit_error_vs_q32',
              'mean_logit_error_vs_q32','sparse_type','status','notes']

with open(results_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
    writer.writeheader()
    writer.writerows(all_results)

print(f'Saved: {results_csv}')
n_total = len(all_results)
print(f'Total: {n_total} results ({len(q16_results)} Q16 + {len(q8_results)} Q8)')

# Print summary table
print(f'\n{"ID":<28} {"Q":<4} {"S%":<5} {"ActD":<5} {"Acc":<8} {"Match":<8} {"F1":<8} {"vs ViT4Mal"}')
print('-' * 95)
for r in all_results:
    acc_pct = r['accuracy'] * 100
    if acc_pct >= 93.89:
        vs = 'ABOVE rec'
    elif acc_pct >= 92.41:
        vs = 'ABOVE fastest'
    else:
        vs = 'BELOW'
    print(f'{r["experiment_id"]:<28} {r["quant_bits"]:<4} {r["sparsity_percent"]:<5} '
          f'{r["active_dim"]:<5} {acc_pct:<8.2f} {r["prediction_match_rate_vs_q32"]:<8.4f} '
          f'{r["f1"]:<8.4f} {vs}')


---
## Cell 10: Conservative HLS Candidate Selection
---

**Rules:**
1. accuracy >= 92.41% (ViT4Mal fastest) → eligible
2. accuracy >= 93.0% → higher priority
3. Q16 S5/S10/S15/S20 are primary targets
4. Q8 only S0/S5/S10, best performer only
5. NO 50%/70%/90%/100% in first HLS batch


In [ ]:
BASE_VIVADO_LUT = 14004  # Q16 S0 PARETO2-AXI Vivado actual

hls_candidates = []

for r in all_results:
    if r['status'] != 'SUCCESS':
        continue
    if r['sparsity_percent'] > 30:
        continue
    
    qbits = r['quant_bits']
    sp = r['sparsity_percent']
    acc = r['accuracy']
    match = r['prediction_match_rate_vs_q32']
    active_dim = r['active_dim']
    
    eligible = acc >= 0.9241
    high_priority = acc >= 0.93 and match >= 0.97
    
    bit_factor = qbits / 16.0
    dim_factor = max(active_dim, 1) / D_MODEL
    est_lut = int(BASE_VIVADO_LUT * bit_factor * dim_factor)
    
    r['hls_eligible'] = eligible
    r['hls_priority'] = 'HIGH' if high_priority else ('MEDIUM' if eligible else 'LOW')
    r['est_vivado_lut'] = est_lut
    r['hls_reason'] = (
        f'acc={acc*100:.2f}%>=92.41%, match={match:.4f}>=0.97'
        if high_priority else
        f'acc={acc*100:.2f}%>=92.41%' if eligible else
        f'acc={acc*100:.2f}%<92.41%'
    )
    
    if eligible:
        hls_candidates.append(r)

# Sort: Q16 first, then accuracy descending
hls_candidates.sort(key=lambda x: (0 if x['quant_bits'] == 16 else 1, -x['accuracy']))

# Select top 3 for first HLS batch (Q16 S5, S10, S15/S20)
first_batch = [c for c in hls_candidates if c['quant_bits'] == 16 and c['sparsity_percent'] <= 20][:3]

# Save candidate CSV
cand_csv = f'{TABLES}/experiment1_attention_only_progressive_hls_candidates.csv'
cand_fields = ['experiment_id','quant_bits','sparsity_percent','active_dim',
               'active_channels','accuracy','prediction_match_rate_vs_q32',
               'hls_eligible','hls_priority','est_vivado_lut','hls_reason']

with open(cand_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=cand_fields, extrasaction='ignore')
    writer.writeheader()
    all_flagged = []
    for r in all_results:
        if r['status'] != 'SUCCESS':
            continue
        row = {k: r.get(k, '') for k in cand_fields}
        all_flagged.append(row)
    writer.writerows(all_flagged)

print(f'Saved: {cand_csv}')
n_elig = len(hls_candidates)
print(f'\nHLS Candidates (eligible, acc >= 92.41%): {n_elig}')
for c in hls_candidates:
    print(f'  [{c["hls_priority"]:<6}] {c["experiment_id"]:<28} '
          f'acc={c["accuracy"]*100:.2f}% match={c["prediction_match_rate_vs_q32"]:.4f} '
          f'est_LUT={c["est_vivado_lut"]}')

n_fb = len(first_batch)
print(f'\nFirst HLS Batch (max 3):')
for c in first_batch:
    print(f'  {c["experiment_id"]:<28} Q={c["quant_bits"]} S={c["sparsity_percent"]}% '
          f'acc={c["accuracy"]*100:.2f}% est_LUT={c["est_vivado_lut"]}')

if n_fb == 0:
    print('  WARNING: No Q16 attention-only points meet 92.41%!')
    print('  Attention-only Q/K sparse may still degrade accuracy too much.')
    print('  Consider even smaller sparsity steps (2%, 3%).')


---
## Cell 11: Comparison: Attention-Only vs Global Pruning
---


In [ ]:
print('=' * 80)
print('COMPARISON: Global Pruning vs Attention-Only Q/K Sparse')
print('=' * 80)

global_results = {}
global_csv = f'{TABLES}/experiment1_software_quant_sparsity_results.csv'
if os.path.exists(global_csv):
    with open(global_csv) as f:
        for row in csv.DictReader(f):
            key = (int(row['quant_bits']), int(row['sparsity_percent']))
            if row['status'] == 'SUCCESS':
                global_results[key] = float(row['accuracy'])

print(f'\n{"Q":<4} {"S%":<5} {"Global Acc":<12} {"Attn-Only Acc":<14} {"Delta":<10} {"Winner"}')
print('-' * 70)

for r in all_results:
    qbits = r['quant_bits']
    sp = r['sparsity_percent']
    attn_acc = r['accuracy']
    global_acc = global_results.get((qbits, sp), None)
    
    if global_acc is not None:
        delta = (attn_acc - global_acc) * 100
        winner = 'ATTN-ONLY' if delta > 0.01 else ('GLOBAL' if delta < -0.01 else 'TIE')
        print(f'{qbits:<4} {sp:<5} {global_acc*100:<12.2f} {attn_acc*100:<14.2f} {delta:+<10.2f}pp {winner}')
    else:
        print(f'{qbits:<4} {sp:<5} {"N/A":<12} {attn_acc*100:<14.2f} {"N/A":<10} —')

print(f'\nPositive delta = attention-only better. Negative = global pruning better.')
print(f'(At S=0%, both methods are identical — no sparsity applied.)')


---
## Cell 12: Summary and Next Steps
---

In [ ]:
print('=' * 70)
print('PROGRESSIVE ATTENTION-ONLY SPARSE — FINAL SUMMARY')
print('=' * 70)

n_q16_above = sum(1 for r in q16_results if r['accuracy'] >= 0.9241)
n_q8_above  = sum(1 for r in q8_results if r['accuracy'] >= 0.9241)

print(f'\nQ16 attention-only: {len(q16_results)} points, {n_q16_above} above ViT4Mal fastest')
print(f'Q8 attention-only:  {len(q8_results)} points, {n_q8_above} above ViT4Mal fastest')
print(f'HLS candidates (acc >= 92.41%): {n_elig}')
print(f'First HLS batch: {n_fb}')

print(f'\nKey findings:')
print(f'1. Q16 S0 (dense, no sparse): acc={q16_acc_0*100:.2f}% — baseline (PYNQ measured)')
if len(q16_results) > 2:
    q16_acc_5  = [r for r in q16_results if r['sparsity_percent'] == 5][0]['accuracy']
    q16_acc_10 = [r for r in q16_results if r['sparsity_percent'] == 10][0]['accuracy']
    print(f'2. Q16 S5:  acc={q16_acc_5*100:.2f}% — drop from S0={(q16_acc_0-q16_acc_5)*100:.2f}pp')
    print(f'3. Q16 S10: acc={q16_acc_10*100:.2f}% — drop from S0={(q16_acc_0-q16_acc_10)*100:.2f}pp')

print(f'\nOutput files:')
print(f'  {results_csv}')
print(f'  {cand_csv}')

print(f'\nNext steps:')
if n_fb > 0:
    names = ', '.join(c['experiment_id'] for c in first_batch)
    print(f'  1. Prepare HLS for: {names}')
    print(f'  2. Run HLS CSIM + CSYNTH for these candidates')
    print(f'  3. Compare real HLS LUT vs Q16 S0 baseline (LUT=14,004)')
else:
    print(f'  WARNING: No candidates meet 92.41% threshold!')
    print(f'  A. Try finer sparsity (2%, 3%, 7%)')
    print(f'  B. Try different channel selection criterion')
    print(f'  C. Accept Q8 dense as low-LUT point, skip sparse for now')

print(f'\nWhat NOT to do now:')
print(f'  - NO 50%/70%/90%/100% sparse HLS')
print(f'  - NO Q4 sparse HLS')
print(f'  - NO Vivado or PYNQ for sparse yet')
print(f'  - NO claiming sparse improves performance (yet)')
print(f'  - NO deleting old results or figures')
